# 05. FastAPI へのエクスポート（本番反映）

`data/staging/` の検証済み成果物を本番パスへプロモーションする。

⚠️ **このノートブックは本番データを変更します。**
実行前に必ず staging の内容を確認してから `PROMOTE=True` に変更してください。

In [ ]:
# ── 重要: デフォルトは PROMOTE=False（確認モード）────────────────────────────
# 本番反映する場合のみ True に変更
PROMOTE = False

import sys, json, logging
from pathlib import Path

try:
    BASE
except NameError:
    BASE        = Path("/Users/takehirosato/Desktop/AI_TradeManagement")
    STAGING_DIR = BASE / "data" / "staging"
    sys.path.insert(0, str(BASE / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print(f"PROMOTE={PROMOTE}  ({'🟢 本番反映モード' if PROMOTE else '🔵 確認モード (dry-run)'})")

## 1. staging 成果物の確認

In [ ]:
from pipeline.export.staging_exporter import staging_summary

summary = staging_summary(STAGING_DIR)
print("staging ディレクトリの現在の成果物:")
print("-" * 60)
for path, info in summary.items():
    size = info.get("size_bytes", 0)
    records = info.get("records", "")
    rec_str = f"  ({records} 件)" if records else ""
    print(f"  {path:<45} {size:>10,} bytes{rec_str}")

## 2. 反映対象の選択

In [ ]:
# 反映するファイルを選択（None = 全て）
# 個別に選ぶ場合は以下のようにリストを指定:
# TARGETS = ["faiss/entities.index", "faiss/entities_meta.json"]
TARGETS = None  # 全て

from pipeline.export.staging_exporter import PROMOTION_TARGETS
print("反映対象マッピング:")
for key, dest in PROMOTION_TARGETS.items():
    staging_file = STAGING_DIR / key
    exists = "✅" if staging_file.exists() else "❌ (未生成)"
    print(f"  {exists} {key}")
    print(f"       → {dest}")

## 3. プロモーション実行

In [ ]:
from pipeline.export.staging_exporter import promote

results = promote(
    staging_dir=STAGING_DIR,
    targets=TARGETS,
    dry_run=not PROMOTE,
)

print("\nプロモーション結果:")
print("-" * 60)
for key, status in results.items():
    icon = "✅" if status == "promoted" else ("🔵" if "DRY_RUN" in status else ("⏭" if "skipped" in status else "❌"))
    print(f"  {icon} {key}: {status}")

## 4. 反映後の動作確認

In [ ]:
if PROMOTE:
    # FastAPI モジュールの再起動が必要な場合のガイド
    print("=" * 60)
    print("✅ プロモーション完了")
    print("=" * 60)
    print()
    print("次のステップ:")
    print("  1. screening モジュールを再起動してFAISSをリロード:")
    print("     pkill -f 'uvicorn.*8005'; cd modules/screening && ...")
    print()
    print("  2. ai_validation の matrix_rules FAISS をリビルド:")
    print("     POST http://localhost:8001/api/admin/faiss/rebuild")
    print()
    print("  3. platform-core でデータセットをリビルド:")
    print("     POST http://localhost:8000/admin/datasets/build")
    print()
    print("  4. 動作確認:")
    print("     curl http://localhost:8005/health")
    print("     curl http://localhost:8001/health")
else:
    print("🔵 確認モード完了。本番反映するには PROMOTE=True に変更して再実行。")